In [1]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
#from Minimal_Model_2D import *
import Minimal_Model_2D as mm2d
import charge_trap_pot_model_4_layers as ctm 
import charge_trap_pot_model as ctm3
import time


np.set_printoptions(threshold=sys.maxsize)

In [2]:
NSM = int(60/2)            # number of sm x lattice points
M = int(60/2)              # should be atleast thrice confinement length
AX = int(5*2)              # lattice spacing in nm
AY = int(5*2)              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized Ge
MEFF_SM = 0.023     # Ge
NSC = int(800/2)
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400        # meV-nm^3 
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 75           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SM = 0.8 + OMEGA/2
MU_SC = 0.8 + OMEGA/2

print('mu = ', MU_SM)

ModCT = ctm.Image_charge_model(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM, t_ox=7, t_sige=48, t_ge=20)
ModCT3 = ctm3.Image_charge_model(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM, t_ox=7, t_sige=48)


lengths= 300 75
subband energies= 1.1789724682017417 3.536917404605225
4t= 66.31720133634798
mu =  1.9789724682017418


In [3]:
NTOT*M 

24900

In [4]:
XCT = (NSC+NSM/2)*AX
ZCT = 7
ZIM = 0   ### immaterial in img charge model 
ZQW = 57


In [5]:
HA =  ModCT.H_sys(PHI, 'cu', XCT, 50, ZCT, ZIM, ZQW, f_ct=1) 



In [6]:
HA.shape

(99600, 99600)

In [7]:
HB = ModCT.H_sys(PHI, 'cu', XCT, 50, ZCT, ZIM, ZQW, f_ct=0) + ModCT.H_charge_trap_sys(XCT, 50, ZCT, ZIM, ZQW) 

In [8]:
HB.shape

(99600, 99600)

In [9]:
import scipy.sparse as Spar

In [10]:
def compare_sparse_matrices(a, b, rtol=1e-7, atol=1e-10):
    if not Spar.issparse(a) or not Spar.issparse(b):
            raise TypeError("Both inputs must be scipy sparse matrices.")

    if a.shape != b.shape:
        return False

    a = a.tocsr(copy=True)
    b = b.tocsr(copy=True)

    a.sum_duplicates()
    b.sum_duplicates()
    a.eliminate_zeros()
    b.eliminate_zeros()
    a.sort_indices()
    b.sort_indices()

    return (
            np.array_equal(a.indptr, b.indptr)
            and np.array_equal(a.indices, b.indices)
            and np.allclose(a.data, b.data, rtol=rtol, atol=atol)
        )

In [14]:
compare_sparse_matrices(HA, HB, 1e-12, 1e-12)

True

In [16]:
diff = HA-HB

In [17]:
if diff.nnz > 0:
    k = np.argmax(np.abs(diff.data))
    max_diff = diff.data[k]

    row, col = diff.nonzero()
    i = row[k]
    j = col[k]

    print("Maximum difference:", max_diff)
    print("Absolute difference:", abs(max_diff))
    print("Location:", i, j)

In [18]:
HA.toarray()[29465, 29465]

(73.38576809940396+0j)

In [19]:
HB.toarray()[29465, 29465]

(73.38576809940396+0j)

In [20]:
import time

In [21]:
#XCT = np.linspace((NSC-10)*AX, (NSC+NSM+10)*AX, 35)
#XCT = np.linspace(0, (NTOT)*AX, 35)
K_NUM = 4 
XCT = (NSC+NSM/2)*AX

# Define the boundary points
a = -100/2*AY
b = 0
c = M*AY
d = (M+100/2)*AY

print(a, b, c ,d)

# Define the number of points between each segment
#n_ab = 50  # number of points between a and b
#n_bc = 101  # number of points between b and c (more points)
#n_cd = 50  # number of points between c and d

n_ab = 2  # number of points between a and b
n_bc = 3  # number of points between b and c (more points)
n_cd = 2  # number of points between c and d




# Create linearly spaced points between each pair
y_ab = np.linspace(a, b, n_ab, endpoint=False)
y_bc = np.linspace(b, c, n_bc, endpoint=False)
y_cd = np.linspace(c, d, n_cd, endpoint=True)

# Concatenate all points
YCT = np.concatenate((y_ab, y_bc, y_cd))

#print(YCT)

print('X co-ordinate of charge trap =', XCT )

ZCT = 7
ZIM = 0   ### immaterial in img charge model 
ZQW = 57

Ep = np.empty((YCT.shape[0],K_NUM))

t0 = time.perf_counter()

H_wo_CT = ModCT.H_sys(PHI, 'cu', XCT, YCT[0], ZCT, ZIM, ZQW, f_ct=0)

t1 = time.perf_counter()

print("time H_wo_CT:  ", t1-t0)

for i in range(YCT.shape[0]):

    t = time.perf_counter()

    print(YCT[i], i)

    H_CT = ModCT.H_charge_trap_sys(XCT, YCT[i], ZCT, ZIM, ZQW)
    
    t2 = time.perf_counter()
    
    H = H_CT + H_wo_CT
    print("here")

    t3 = time.perf_counter()

    print("time: ", t2-t, t3-t2)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)

    Ep[i,:] = Es

    #if i %10 ==0 :
    if i %1 ==0 :
        print(Ep[i,:])

#np.save('cubic_E_vs_yct_image_chrg_model.npy', Ep)
np.save('cubic_E_vs_yct_image_chrg_model_full_4_layers_limited_pts_V3.npy', Ep)

-500.0 0 300 800.0
X co-ordinate of charge trap = 4150.0
time H_wo_CT:   127.16025589965284
-500.0 0
here
time:  28.945786099880934 0.048856799490749836
[-0.07568224 -0.07538521  0.07538521  0.07568224]
-250.0 1
here
time:  29.728965299203992 0.04212220013141632
[-0.07565787 -0.07536105  0.07536105  0.07565787]
0.0 2
here
time:  28.022418700158596 0.04339639935642481
[-0.07520383 -0.07491051  0.07491051  0.07520383]
100.0 3


KeyboardInterrupt: 

In [ ]:
#XCT = np.linspace((NSC-10)*AX, (NSC+NSM+10)*AX, 35)
#XCT = np.linspace(0, (NTOT)*AX, 35)

K_NUM = 4 

XCT = (NSC+NSM/2)*AX

# Define the boundary points
a = -100/2*AY
b = 0
c = M*AY
d = (M+100/2)*AY

print(a, b, c ,d)

# Define the number of points between each segment
#n_ab = 50  # number of points between a and b
#n_bc = 101  # number of points between b and c (more points)
#n_cd = 50  # number of points between c and d

n_ab = 2  # number of points between a and b
n_bc = 3  # number of points between b and c (more points)
n_cd = 2  # number of points between c and d




# Create linearly spaced points between each pair
y_ab = np.linspace(a, b, n_ab, endpoint=False)
y_bc = np.linspace(b, c, n_bc, endpoint=False)
y_cd = np.linspace(c, d, n_cd, endpoint=True)

# Concatenate all points
YCT = np.concatenate((y_ab, y_bc, y_cd))

#print(YCT)

print('X co-ordinate of charge trap =', XCT )

ZCT = 7
ZIM = 0   ### immaterial in img charge model 
ZQW = 57

Ep = np.empty((YCT.shape[0],K_NUM))

for i in range(YCT.shape[0]):

    print(YCT[i], i)

    H = ModCT.H_sys(PHI, 'cu', XCT, YCT[i], ZCT, ZIM, ZQW, f_ct=1)
    print("here")

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)

    Ep[i,:] = Es

    #if i %10 ==0 :
    if i %1 ==0 :
        print(Ep[i,:])

#np.save('cubic_E_vs_yct_image_chrg_model.npy', Ep)
np.save('cubic_E_vs_yct_image_chrg_model_full_4_layers_limited_pts_v2.npy', Ep)

In [ ]:
#Ep_cu_mm2d = np.load('cubic_E_vs_yct_Zdiff_14_wider_yct_width_75nm_refined.npy')
#XCT = np.linspace((NSC-10)*AX, (NSC+NSM+10)*AX, 35)
#XCT = np.linspace(0, (NTOT)*AX, 35)
XCT = (NSC+NSM/2)*AX

# Define the boundary points
a = -100*AY
b = 0
c = M*AY
d = (M+100)*AY

# Define the number of points between each segment
n_ab = 50  # number of points between a and b
n_bc = 101  # number of points between b and c (more points)
n_cd = 50  # number of points between c and d

#n_ab = 5  # number of points between a and b
#n_bc = 11  # number of points between b and c (more points)
#n_cd = 5  # number of points between c and d




# Create linearly spaced points between each pair
y_ab = np.linspace(a, b, n_ab, endpoint=False)
y_bc = np.linspace(b, c, n_bc, endpoint=False)
y_cd = np.linspace(c, d, n_cd, endpoint=True)

# Concatenate all points
YCT_full = np.concatenate((y_ab, y_bc, y_cd))



# Define the boundary points
a = -100*AY
b = 0
c = M*AY
d = (M+100)*AY

# Define the number of points between each segment
#n_ab = 50  # number of points between a and b
#n_bc = 101  # number of points between b and c (more points)
#n_cd = 50  # number of points between c and d

n_ab = 2  # number of points between a and b
n_bc = 3  # number of points between b and c (more points)
n_cd = 2  # number of points between c and d




# Create linearly spaced points between each pair
y_ab = np.linspace(a, b, n_ab, endpoint=False)
y_bc = np.linspace(b, c, n_bc, endpoint=False)
y_cd = np.linspace(c, d, n_cd, endpoint=True)

# Concatenate all points
YCT_limited = np.concatenate((y_ab, y_bc, y_cd))
#print(YCT)

print('X co-ordinate of charge trap =', XCT )

ZCT = 7
ZIM = 0   ### immaterial in img charge model 
ZQW = 57


Ep_cu_ctm = np.load('cubic_E_vs_yct_image_chrg_model_full.npy')

Ep_cu_ctm4 = np.load('cubic_E_vs_yct_image_chrg_model_full_4_layers_limited_pts.npy')

In [ ]:
# Set the font family to Arial
plt.rcParams['font.family'] = 'Arial'

plt.rc('font', size=24)

f = plt.figure(figsize=(10, 8))

ax = plt.axis()

cubic_spin_split = Ep_cu_ctm[:, K_NUM//2+1]-Ep_cu_ctm[:, K_NUM//2]
cubic_spin_split4 = Ep_cu_ctm4[:, K_NUM//2+1]-Ep_cu_ctm4[:, K_NUM//2]
#cubic_spin_split = Ep[:, K_NUM//2+1]-Ep[:, K_NUM//2] 

#min_spin_split = np.min(cubic_spin_split)

max_spin_split = np.max(cubic_spin_split)
max_spin_split4 = np.max(cubic_spin_split4)


Ep_wct = cubic_spin_split/max_spin_split
Ep_wct4 = cubic_spin_split4/max_spin_split4

plt.scatter(YCT_full, Ep_wct )
plt.scatter(YCT_limited, Ep_wct4 )
#plt.axvline(x=NSC*AX, color='black', linestyle='dotted')
#plt.axvline(x=(NSC+NSM-1)*AX, color='black', linestyle='dotted')

plt.axvline(x=0, color='black', linestyle='dotted')
plt.axvline(x=M*AY, color='black', linestyle='dotted')
plt.axvline(x=M/2*AY, color='black', linestyle='dotted')
plt.axvline(x=100, color='black', linestyle='dotted')
plt.axvline(x=200, color='black', linestyle='dotted')

plt.xlabel(r'y  co-ordinate of charge trap, $y_{ct}$ (nm)', fontsize=32)
plt.ylabel(r'$\dfrac{E_s}{E_{s0}}$', fontsize=32)

In [ ]:
#XCT = np.linspace((NSC-10)*AX, (NSC+NSM+10)*AX, 35)
#XCT = np.linspace(0, (NTOT)*AX, 35)
###

XCT = (NSC+NSM/2)*AX

# Define the boundary points
a = -100*AY
b = 0
c = M*AY
d = (M+100)*AY

# Define the number of points between each segment
#n_ab = 50  # number of points between a and b
#n_bc = 101  # number of points between b and c (more points)
#n_cd = 50  # number of points between c and d

n_ab = 2  # number of points between a and b
n_bc = 3  # number of points between b and c (more points)
n_cd = 2  # number of points between c and d




# Create linearly spaced points between each pair
y_ab = np.linspace(a, b, n_ab, endpoint=False)
y_bc = np.linspace(b, c, n_bc, endpoint=False)
y_cd = np.linspace(c, d, n_cd, endpoint=True)

# Concatenate all points
YCT = np.concatenate((y_ab, y_bc, y_cd))

#print(YCT)

print('X co-ordinate of charge trap =', XCT )

ZCT = 7
ZIM = 0   ### immaterial in img charge model 
ZQW = 57

Ep = np.empty((YCT.shape[0],4))

for i in range(YCT.shape[0]):

    print(YCT[i], i)

    H = ModCT3.H_sys(PHI, 'cu', XCT, YCT[i], ZCT, ZIM, ZQW, f_ct=1)
    print("here")

    E = eigsh(H, k=4,sigma=0, return_eigenvectors=False )

    Es = np.sort(E)

    Ep[i,:] = Es

    #if i %10 ==0 :
    if i %1 ==0 :
        print(Ep[i,:])

#np.save('cubic_E_vs_yct_image_chrg_model.npy', Ep)
#np.save('cubic_E_vs_yct_image_chrg_model_full_4_layers_limited_pts.npy', Ep)

In [ ]:
print(YCT)

In [ ]:
print(YCT[np.argmin(Ep_wct)])

make sure to inclue center point 150 in YCT

In [ ]:
NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized Ge
MEFF_SM = 0.023     # Ge
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400        # meV-nm^3 
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 75           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SM = 0.8 + OMEGA/2
MU_SC = 0.8 + OMEGA/2

print('mu = ', MU_SM)

ModCT = ctm.Image_charge_model(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM, t_ox=7, t_sige=48)

#XCT = np.linspace((NSC-10)*AX, (NSC+NSM+10)*AX, 35)
#XCT = np.linspace(0, (NTOT)*AX, 35)
XCT = (NSC+NSM/2)*AX

##checking teh minimum for small range first 

YCT = np.array([148.51485149 , 150, 151.48514851, 154.45544554]) 

#print(YCT)

print('X co-ordinate of charge trap =', XCT )

ZCT = 7
ZIM = 0   ### immaterial in img charge model 
ZQW = 57

Ep = np.empty((YCT.shape[0],K_NUM))

for i in range(YCT.shape[0]):

    H = ModCT.H_sys(PHI, 'cu', XCT, YCT[i], ZCT, ZIM, ZQW, f_ct=1)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)

    Ep[i,:] = Es

    #if i %10 ==0 :
    if i %1 ==0 :
        print(Ep[i,:])

#np.save('cubic_E_vs_yct_image_chrg_model.npy', Ep)
np.save('cubic_E_vs_yct_image_chrg_model_centering_check.npy', Ep)

In [ ]:
# Set the font family to Arial
plt.rcParams['font.family'] = 'Arial'

plt.rc('font', size=24)

f = plt.figure(figsize=(10, 8))

ax = plt.axis()

cubic_spin_split = Ep[:, K_NUM//2+1]-Ep[:, K_NUM//2] 

min_spin_split = np.min(cubic_spin_split)

max_spin_split = np.max(cubic_spin_split)


Ep_wct = (Ep[:, K_NUM//2+1]-Ep[:, K_NUM//2])/max_spin_split

plt.scatter(YCT, Ep_wct )
#plt.axvline(x=NSC*AX, color='black', linestyle='dotted')
#plt.axvline(x=(NSC+NSM-1)*AX, color='black', linestyle='dotted')

plt.axvline(x=0, color='black', linestyle='dotted')
plt.axvline(x=M*AY, color='black', linestyle='dotted')
plt.axvline(x=M/2*AY, color='black', linestyle='dotted')

plt.xlabel(r'y  co-ordinate of charge trap, $y_{ct}$ (nm)', fontsize=32)
plt.ylabel(r'$\dfrac{E_s}{E_{s0}}$', fontsize=32)
plt.savefig('charge_trap_ES_center_check.png')
#plt.show()